# One-dimensional inverse-bremsstrahlung deposition

This notebook follows one beam through a uniform underdense plasma slab. It compares the ray attenuation and grid-integrated heating with the closed-form solution, plots the local and cumulative deposited power, and separates solver, sheet-sampling, boundary, and grid-scattering errors.

The notebook is intentionally stored without outputs. A full run includes several JAX compilations for the convergence sweeps.

In [ ]:
import sys
import time
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

repo_root = Path.cwd()
if not (repo_root / "configs").is_dir():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from examples._workflows import (
    UNIFORM_1D_CONFIG,
    run_simulation,
    run_uniform_1d,
)
from pyGATH.fields import grid_cell_volumes
from pyGATH.io import load_simulation_config
from pyGATH.raytracing import SPEED_OF_LIGHT, critical_density

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})

## Analytical reference

For constant electron density, temperature, and refractive index $n_r$, the paper's critical-density collision-frequency convention gives

$$\alpha = \frac{2\nu_{ei,c}}{c\,n_r}\left(\frac{n_e}{n_c}\right)^2,$$

where $dP/dx=-\alpha P$. Therefore $P(x)=P_0e^{-\alpha x}$, $q(x)=\alpha P(x)/A$, and the total absorbed fraction across length $L$ is $1-e^{-\alpha L}$. The deck uses $n_e/n_c=0.1$, $L=40\,\mu$m, and $\nu_{ei,c}=206.45$ ps$^{-1}$.

In [ ]:
simulation_1d = load_simulation_config(UNIFORM_1D_CONFIG)
started = time.perf_counter()
trace_1d, grid_1d, beams_1d, deposition_1d, checks_1d = run_uniform_1d()
elapsed = time.perf_counter() - started

print(f"run time: {elapsed:.2f} s")
print(f"terminated cleanly: {bool(trace_1d.terminated)}")
print(f"analytical absorption: {checks_1d['analytical_absorption_fraction']:.8%}")
print(f"direct ray absorption: {checks_1d['direct_absorption_fraction']:.8%}")
print(f"grid-integrated absorption: {checks_1d['deposited_absorption_fraction']:.8%}")
print(
    f"grid scatter residual / incident: {checks_1d['deposition_conservation_error_fraction']:.3e}"
)
print(f"CBET depth: {checks_1d['maximum_cbet_depth']:.1f}")

## Local and cumulative deposition

The first panel compares the cell-centred volumetric heating with $q(x)$. The second integrates cell power from the entrance to show where the total absorption accumulates. Small edge deficits arise because the event-located sheet endpoints can lie an ulp outside the closed grid: their sampled source is zero, and linear simplicial interpolation spreads that zero over half of the adjacent sample interval. This error falls with sheet sampling.

In [ ]:
omega = float(beams_1d.omega[0])
density = float(grid_1d.hydro.ne[0, 0, 0])
density_ratio = density / float(critical_density(omega))
refractive_index = np.sqrt(1.0 - density_ratio)
collision_frequency = (
    simulation_1d.physics.inverse_bremsstrahlung.critical_collision_frequency_hz
)
alpha = 2.0 * collision_frequency / SPEED_OF_LIGHT * density_ratio**2 / refractive_index
x0 = float(grid_1d.xb[0])
x_um = (np.asarray(grid_1d.xc) - x0) * 1e6
incident_intensity = checks_1d["incident_power_w"] / grid_1d.inactive_measure
analytical_density = (
    incident_intensity * alpha * np.exp(-alpha * (np.asarray(grid_1d.xc) - x0))
)
numerical_density = np.asarray(deposition_1d.power_density)[:, 0, 0]
cumulative_numerical = (
    np.cumsum(np.asarray(deposition_1d.cell_power)[:, 0, 0])
    / checks_1d["incident_power_w"]
)
cumulative_analytical = 1.0 - np.exp(-alpha * (np.asarray(grid_1d.xb[1:]) - x0))

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axes[0].plot(x_um, analytical_density, label="analytic", lw=2)
axes[0].plot(x_um, numerical_density, "--", label="grid deposition")
axes[0].set(
    xlabel="distance into slab (µm)", ylabel="deposited power density (W m$^{-3}$)"
)
axes[0].legend()
axes[1].plot(x_um, cumulative_analytical, label="analytic", lw=2)
axes[1].plot(x_um, cumulative_numerical, "--", label="grid cumulative")
axes[1].set(xlabel="distance into slab (µm)", ylabel="absorbed fraction")
axes[1].legend();

In [ ]:
volumes = np.asarray(grid_cell_volumes(grid_1d))[:, 0, 0]
analytic_cell_power = analytical_density * volumes
profile_l1_error = np.sum(
    np.abs(np.asarray(deposition_1d.cell_power)[:, 0, 0] - analytic_cell_power)
) / (checks_1d["analytical_absorption_fraction"] * checks_1d["incident_power_w"])
ray_error = (
    checks_1d["direct_absorption_fraction"]
    - checks_1d["analytical_absorption_fraction"]
)
sheet_integration_error = (
    checks_1d["source_absorption_fraction"] - checks_1d["direct_absorption_fraction"]
)
scatter_error = (
    checks_1d["deposited_absorption_fraction"] - checks_1d["source_absorption_fraction"]
)
print(f"direct-ray minus analytic: {ray_error:+.3e}")
print(f"sheet-source minus direct ray: {sheet_integration_error:+.3e}")
print(f"grid scatter minus sheet source: {scatter_error:+.3e}")
print(f"cell-power L1 profile error: {profile_l1_error:.3e}")

## Error budget

1. **Absorption model:** this example uses the paper's specified $\nu_{ei,c}$ law, not the default NRL Coulomb-log model. Comparing the two is a model comparison, not a discretization error.
2. **ODE/event error:** adaptive ray integration and the exit event set the terminal optical depth. This is measured by direct-ray minus analytical absorption.
3. **Sheet resampling:** deposition is linear between a finite number of sheet samples. The zero-valued just-outside endpoints give the leading 1-D integration error.
4. **Grid representation:** cell-centred power density is a finite-volume average. Pointwise profile error changes with hydro-cell width even when total conservative scatter does not.
5. **Conservative scatter:** `source_power = deposited_power + outside_power` should hold to floating-point precision. This isolates scattering from source reconstruction.
6. **Reduced-dimensional normalization:** the configured inactive area converts W/m$^3$ into total watts. Mismatched inactive lengths would produce a normalization error.
7. **CBET:** it is disabled; a nonzero CBET depth would invalidate this analytical comparison.

## Sheet-sampling convergence

This sweep changes only `nsamples_per_sheet`. It should leave direct ray loss nearly fixed while the grid-integrated source approaches the analytical total. Each distinct shape causes a JAX compilation.

In [ ]:
sample_counts = np.array([129, 257, 513, 1025, 2049])
sampling_rows = []
for count in sample_counts:
    if count == simulation_1d.raytracing.nsamples_per_sheet:
        current_checks = checks_1d
    else:
        varied = replace(
            simulation_1d,
            raytracing=replace(simulation_1d.raytracing, nsamples_per_sheet=int(count)),
        )
        *_, current_checks = run_simulation(varied)
    sampling_rows.append(
        (
            current_checks["direct_absorption_fraction"],
            current_checks["deposited_absorption_fraction"],
        )
    )
sampling_rows = np.asarray(sampling_rows)
reference = checks_1d["analytical_absorption_fraction"]
fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
ax.loglog(
    sample_counts, np.abs(sampling_rows[:, 0] - reference), "o-", label="direct ray"
)
ax.loglog(
    sample_counts, np.abs(sampling_rows[:, 1] - reference), "s-", label="grid integral"
)
ax.set(xlabel="samples per sheet", ylabel="absolute absorption error")
ax.legend();

## ODE-tolerance convergence

This sweep fixes sheet sampling at 513 points and varies only the adaptive solver tolerance. The direct-ray curve isolates propagation and exit-event error from deposition interpolation.

In [ ]:
relative_tolerances = np.array([1e-4, 1e-5, 1e-6, 1e-7])
tolerance_absorption = []
for tolerance in relative_tolerances:
    varied = replace(
        simulation_1d,
        raytracing=replace(
            simulation_1d.raytracing,
            nsamples_per_sheet=513,
            rtol=float(tolerance),
            atol=float(tolerance * 1e-2),
        ),
    )
    *_, current_checks = run_simulation(varied)
    tolerance_absorption.append(current_checks["direct_absorption_fraction"])
tolerance_absorption = np.asarray(tolerance_absorption)
fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
ax.loglog(relative_tolerances, np.abs(tolerance_absorption - reference), "o-")
ax.invert_xaxis()
ax.set(xlabel="relative solver tolerance", ylabel="absolute direct-absorption error");

## Interpretation

The most important separation is between ray attenuation, reconstruction of a continuous source from sampled sheets, and conservative transfer of that source to cells. The analytical result validates the first; sampling convergence validates the second; the conservation residual validates the third. A small pointwise edge error is therefore not evidence of lost power in the scatter operation.